In [1]:
import os
import sys
import csv
import argparse
import psycopg2
import pandas as pd
from psycopg2 import sql
from dotenv import load_dotenv
import csv
from psycopg2.extras import RealDictCursor
# Load .env from project root
load_dotenv()

True

# Connect database

In [2]:
# Env keys (fallbacks)
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME", "postgres")
DB_USER = os.getenv("DB_USER", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
CONNECT_TIMEOUT = int(os.getenv("DB_CONNECT_TIMEOUT", "10"))

In [3]:
def get_db_connection():
    """
    Trả về psycopg2 connection sử dụng biến môi trường từ .env.
    """
    conn_kwargs = {
        "host": DB_HOST,
        "port": DB_PORT,
        "dbname": DB_NAME,
        "user": DB_USER,
        "password": DB_PASSWORD,
        "connect_timeout": CONNECT_TIMEOUT,
    }
    return psycopg2.connect(cursor_factory=RealDictCursor, **conn_kwargs)

def test_connection(sql_query: str = "SELECT now() AS now"):
    """
    Thực thi 1 câu SQL test và in kết quả.
    """
    conn = None
    try:
        conn = get_db_connection()
        with conn.cursor() as cur:
            cur.execute(sql_query)
            rows = cur.fetchall()
            print(f"✓ Query executed: {sql_query}")
            for row in rows:
                print(row)
    except Exception as e:
        print(f"DB error: {e}")
    finally:
        if conn:
            conn.close()

def list_tables():
    """Liệt kê các bảng hiện có (bỏ schema hệ thống)."""
    sql_query = """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type='BASE TABLE'
      AND table_schema NOT IN ('pg_catalog','information_schema')
    ORDER BY table_schema, table_name;
    """
    test_connection(sql_query)

In [4]:
list_tables()

✓ Query executed: 
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type='BASE TABLE'
      AND table_schema NOT IN ('pg_catalog','information_schema')
    ORDER BY table_schema, table_name;
    
RealDictRow([('table_schema', 'public'), ('table_name', 'Action')])
RealDictRow([('table_schema', 'public'), ('table_name', 'Admin')])
RealDictRow([('table_schema', 'public'), ('table_name', 'Answer')])
RealDictRow([('table_schema', 'public'), ('table_name', 'AnswerTranslate')])
RealDictRow([('table_schema', 'public'), ('table_name', 'Attachment')])
RealDictRow([('table_schema', 'public'), ('table_name', 'AttachmentReference')])
RealDictRow([('table_schema', 'public'), ('table_name', 'BiometricDevice')])
RealDictRow([('table_schema', 'public'), ('table_name', 'Category')])
RealDictRow([('table_schema', 'public'), ('table_name', 'CategoryOpenTime')])
RealDictRow([('table_schema', 'public'), ('table_name', 'CategoryOpenTimeTranslate')])
RealDictRow([('table_

In [5]:
# Mở connection
conn = get_db_connection()
cur = conn.cursor()

In [6]:
cur.execute('SELECT * FROM "Poi"')
rows = cur.fetchall()
for poi in rows[:10]:
    print(poi)

RealDictRow([('id', '0f9d2009-9436-46a4-b354-b0261898a39e'), ('created_at', datetime.datetime(2025, 12, 6, 3, 43, 45, 85061)), ('updatedAt', datetime.datetime(2025, 12, 6, 3, 43, 45, 85061)), ('deletedAt', None), ('cityId', '3002ffe3-7c9e-4091-85da-819452858810'), ('source', 'GoogleMaps'), ('content', '{"name":"The Pub Coffee - Beer & Cocktail","placeId":"ChIJ7R9GGLQndTERJRircGTWzn4","description":"","lat":10.8294811,"long":106.7737852,"type":["Cafe","Bar"],"address":"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thành phố Hồ Chí Minh, Vietnam","city":"Ho Chi Minh City","imageUrl":"pois/images/ChIJ7R9GGLQndTERJRircGTWzn4/0d6d8596-ded7-4e10-954b-096aa2783725.jpg","photoUrls":["pois/images/ChIJ7R9GGLQndTERJRircGTWzn4/0d6d8596-ded7-4e10-954b-096aa2783725.jpg","pois/images/ChIJ7R9GGLQndTERJRircGTWzn4/5f9da691-9b82-4ff2-86ea-7316f6f34028.jpg","pois/images/ChIJ7R9GGLQndTERJRircGTWzn4/7b15a9c4-8981-4255-be10-bd193951b4e2.jpg","pois/images/ChIJ7R9GGLQndTERJRircGTWzn4/95c293dc-86be-4fa0-b399-fed54

# Các feature cần phải lấy ra từ table Poi

In [40]:
# id UUID PRIMARY KEY, lấy từ cột id
# name TEXT NOT NULL, lấy từ cột content
# address TEXT, lấy từ cột content 
# lat DOUBLE PRECISION NOT NULL, lấy từ cột content
# lon DOUBLE PRECISION NOT NULL, lấy từ cột content 
# geom GEOMETRY(Point, 4326) NOT NULL, tạo ra code
# poi_type TEXT, lấy từ cột content 
# stay_time DOUBLE PRECISION CHECK (stay_time >= 0), default 30
# avg_stars DOUBLE PRECISION CHECK (avg_stars BETWEEN 0 AND 5), lấy từ cột raw_data
# total_reviews INTEGER CHECK (total_reviews >= 0), từ cột raw_data
# normalize_stars_reviews DOUBLE PRECISION # normalize avg_stars và total_reviews về thang điểm 0-1

# Tạo 1 file csv

In [ ]:
file_path = os.path.join(os.getcwd(), "data/data.csv")
print(f"File path: {file_path}")

# Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(file_path), exist_ok=True)

# Tạo file nếu chưa tồn tại
if not os.path.exists(file_path):
    with open(file_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

# Đọc dữ liệu
if not os.path.exists(file_path) or os.stat(file_path).st_size == 0:
    df = pd.DataFrame()   # df rỗng, chưa có cột
else:
    df = pd.read_csv(file_path)

print(df)

File path: c:\Users\nguye\Desktop\vinamo\Kyanon-support-localtion\scripts\clean_data\data/data1.csv
Empty DataFrame
Columns: []
Index: []


# Trích xuất các trường

In [10]:
# id
id = []
cur.execute('SELECT id FROM "Poi"')
rows = cur.fetchall()
for poi in rows:
    id.append(poi['id'])
df["id"] = id

In [ ]:
import json

name = []
address = []
lat = []
lon = []
poi_type = []

cur.execute('SELECT content FROM "Poi"')
rows = cur.fetchall()

for poi in rows:
    raw = poi.get("content")

    if not isinstance(raw, str):
        continue

    try:
        content = json.loads(raw)  # prase_json
    except Exception as e:
        print("Invalid JSON:", raw)
        continue

    name.append(content.get("name"))
    address.append(content.get("address"))
    lat.append(content.get("lat"))
    lon.append(content.get("long"))
    types = content.get("type")
    if isinstance(types, list):
        poi_type.append(",".join(types))
    else:
        poi_type.append(types)

df["name"] = name
df["address"] = address
df["lat"] = lat
df["lon"] = lon
df["poi_type"] = poi_type

Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
Raw content type: <class 'str'>
end=
R

In [44]:
# avg_stars và total_reviews từ raw_data
avg_stars = []
total_reviews = []
cur.execute('SELECT raw_data FROM "Poi"')
rows = cur.fetchall()
for poi in rows:
    raw_data = poi.get("raw_data")
    google = raw_data.get("google")
    avg_stars.append(google.get("totalScore"))
    total_reviews.append(google.get("reviewsCount"))
df["avg_stars"] = avg_stars
df["total_reviews"] = total_reviews

# Xuất vào csv

In [45]:
# Kiểm tra dữ liệu hiện tại của avg_stars và total_reviews
print("Kiểm tra dữ liệu avg_stars và total_reviews:")
print(f"Kiểu dữ liệu avg_stars: {df['avg_stars'].dtype}")
print(f"Kiểu dữ liệu total_reviews: {df['total_reviews'].dtype}")
print(f"Số lượng giá trị null trong avg_stars: {df['avg_stars'].isna().sum()}")
print(f"Số lượng giá trị null trong total_reviews: {df['total_reviews'].isna().sum()}")
print(f"\nMẫu dữ liệu:")
print(df[['name', 'avg_stars', 'total_reviews']].head(15))
print(f"\nCác giá trị unique của avg_stars: {df['avg_stars'].unique()[:20]}")
print(f"\nCác giá trị unique của total_reviews: {df['total_reviews'].unique()[:20]}")

Kiểm tra dữ liệu avg_stars và total_reviews:
Kiểu dữ liệu avg_stars: object
Kiểu dữ liệu total_reviews: object
Số lượng giá trị null trong avg_stars: 2
Số lượng giá trị null trong total_reviews: 0

Mẫu dữ liệu:
                                                 name avg_stars total_reviews
0                    The Pub Coffee - Beer & Cocktail       4.9           181
1                                             Julieta       4.3          2053
2                  BAIETA Restaurant Saigon Thao Dien       4.5           285
3                           Parroquia San Felipe Neri       4.7           329
4                                              BÀ BAR       4.8           182
5                                    Art Bar Viet Nam       4.9           165
6   Ngỡ Cafe | Cà phê view đẹp Thủ Đức | Coffee ng...       4.7           216
7                              TEEMAY COFFEE ROASTERS         5           275
8                        La Mafia se sienta a la mesa       4.3          1322
9        

In [46]:
# Xử lý avg_stars và total_reviews theo yêu cầu
# Nếu avg_stars hoặc total_reviews không có giá trị hoặc 1 trong 2 có giá trị 0
# thì cả 2 trường sẽ được đưa về giá trị thấp nhất

# Chuyển đổi sang numeric, các giá trị không hợp lệ sẽ thành NaN
df['avg_stars'] = pd.to_numeric(df['avg_stars'], errors='coerce')
df['total_reviews'] = pd.to_numeric(df['total_reviews'], errors='coerce')

# Tìm giá trị thấp nhất của mỗi cột (không tính null và 0)
min_avg_stars = df[(df['avg_stars'].notna()) & (df['avg_stars'] > 0)]['avg_stars'].min()
min_total_reviews = df[(df['total_reviews'].notna()) & (df['total_reviews'] > 0)]['total_reviews'].min()

print(f"Giá trị thấp nhất của avg_stars (không tính null và 0): {min_avg_stars}")
print(f"Giá trị thấp nhất của total_reviews (không tính null và 0): {min_total_reviews}")

# Tạo mask để xác định các hàng cần cập nhật
# Điều kiện: avg_stars hoặc total_reviews là null, hoặc 1 trong 2 bằng 0
mask = (
    df['avg_stars'].isna() | 
    df['total_reviews'].isna() | 
    (df['avg_stars'] == 0) | 
    (df['total_reviews'] == 0)
)

print(f"\nSố lượng hàng cần cập nhật: {mask.sum()}")
print(f"Trước khi cập nhật:")
print(df[mask][['name', 'avg_stars', 'total_reviews']].head(15))

# Cập nhật cả 2 cột về giá trị thấp nhất
df.loc[mask, 'avg_stars'] = min_avg_stars
df.loc[mask, 'total_reviews'] = min_total_reviews

print(f"\nSau khi cập nhật:")
print(df[mask][['name', 'avg_stars', 'total_reviews']].head(15))

print(f"\nKiểm tra lại:")
print(f"Số lượng giá trị null trong avg_stars: {df['avg_stars'].isna().sum()}")
print(f"Số lượng giá trị null trong total_reviews: {df['total_reviews'].isna().sum()}")
print(f"Số lượng giá trị 0 trong avg_stars: {(df['avg_stars'] == 0).sum()}")
print(f"Số lượng giá trị 0 trong total_reviews: {(df['total_reviews'] == 0).sum()}")

Giá trị thấp nhất của avg_stars (không tính null và 0): 3.0
Giá trị thấp nhất của total_reviews (không tính null và 0): 1

Số lượng hàng cần cập nhật: 10
Trước khi cập nhật:
                                                   name  avg_stars  \
255                                Co.opmart Chu Van An        4.1   
257                        PICKLEBALL COLEGIO LA COLINA        NaN   
308                         GOPICK - 250 Lương Định Của        4.6   
597                            Tân Định Catholic Church        4.4   
681                      Nhim Sau Rieng Gelato & Coffee        4.8   
716                             Café Slow in the Garden        4.8   
746                                        Chợ Hạt Điều        4.2   
827                                        Ăn Vặt Bé Bơ        5.0   
877   BIA HƠI HÀ NỘI ĐẠI VIỆT | QUÁN BIA HƠI QUẬN 12...        3.8   
1452                                    parque infantil        NaN   

      total_reviews  
255               0  
257        

In [47]:
df.to_csv(file_path, index=False)

# Thêm các feature

In [ ]:
# Crowd,Children,Offerings,Atmosphere,Highlights,Popular for,Dining options, Accessibility options lấy từ metadata.additionalInfo

In [8]:
cur.execute('SELECT metadata FROM "Poi"')
rows = cur.fetchall()
for poi in rows[:10]:
    print(poi)

RealDictRow([('metadata', {'city': 'Ho Chi Minh City', 'poiType': ['Cafe', 'Bar'], 'totalScore': 4.9, 'imagesCount': 55, 'openingHours': [{'day': 'Monday', 'hours': 'Open 24 hours'}, {'day': 'Tuesday', 'hours': 'Open 24 hours'}, {'day': 'Wednesday', 'hours': 'Open 24 hours'}, {'day': 'Thursday', 'hours': 'Open 24 hours'}, {'day': 'Friday', 'hours': 'Open 24 hours'}, {'day': 'Saturday', 'hours': 'Open 24 hours'}, {'day': 'Sunday', 'hours': 'Open 24 hours'}], 'reviewsCount': 181, 'additionalInfo': {'Crowd': [{'Groups': True}], 'Payments': [{'NFC mobile payments': True}], 'Planning': [{'Accepts reservations': True}], 'Amenities': [{'Bar onsite': True}, {'Restroom': True}], 'Offerings': [{'Alcohol': True}, {'Beer': True}, {'Cocktails': True}, {'Coffee': True}, {'Hard liquor': True}], 'Atmosphere': [{'Casual': True}, {'Cozy': True}], 'Highlights': [{'Great beer selection': True}, {'Great coffee': True}, {'Live music': True}, {'Live performances': True}], 'Dining options': [{'Table service':

In [9]:
def extract_true_keys(items):
    if not isinstance(items, list):
        return []
    return [
        key
        for d in items if isinstance(d, dict)
        for key, value in d.items()
        if value is True
    ]


In [10]:
crowd = []
offerings = []
atmosphere = []
highlights = []
dining_options = []
children = []
accessibility = []
popular_for = []

for poi in rows:
    metadata = poi.get("metadata") or {}
    additional = metadata.get("additionalInfo") or {}

    if not isinstance(additional, dict):
        continue

    crowd.append(extract_true_keys(additional.get("Crowd")))
    offerings.append(extract_true_keys(additional.get("Offerings")))
    atmosphere.append(extract_true_keys(additional.get("Atmosphere")))
    highlights.append(extract_true_keys(additional.get("Highlights")))
    dining_options.append(extract_true_keys(additional.get("Dining options")))
    children.append(extract_true_keys(additional.get("Children")))
    accessibility.append(extract_true_keys(additional.get("Accessibility")))
    popular_for.append(extract_true_keys(additional.get("Popular for")))


In [11]:
print("Crowd:", len(crowd))
print("Offerings:", len(offerings))
print("Atmosphere:", len(atmosphere))
print("Highlights:", len(highlights))
print("Dining options:", len(dining_options))
print("Children:", len(children))
print("Accessibility:", len(accessibility))
print("Popular for:", len(popular_for))


Crowd: 1454
Offerings: 1454
Atmosphere: 1454
Highlights: 1454
Dining options: 1454
Children: 1454
Accessibility: 1454
Popular for: 1454


In [12]:
print(crowd[:10])
print(offerings[:10])
print(atmosphere[:10])
print(highlights[:10])
print(dining_options[:10])
print(children[:10])
print(accessibility[:10])
print(popular_for[:10])

[['Groups'], ['College students', 'Groups', 'Tourists'], ['Family-friendly', 'Groups', 'LGBTQ+ friendly', 'Tourists'], [], ['Groups', 'LGBTQ+ friendly', 'Transgender safespace'], ['Groups', 'LGBTQ+ friendly', 'Tourists', 'Transgender safespace'], ['College students', 'Groups'], ['College students', 'Groups'], ['Family-friendly', 'Groups', 'Tourists'], ['Groups', 'LGBTQ+ friendly', 'Transgender safespace']]
[['Alcohol', 'Beer', 'Cocktails', 'Coffee', 'Hard liquor'], ['Alcohol', 'Beer', 'Coffee', 'Healthy options', 'Organic dishes', 'Prepared foods', 'Quick bite', 'Vegan options', 'Vegetarian options', 'Wine'], ['Alcohol', 'Beer', 'Cocktails', 'Coffee', 'Happy hour drinks', 'Hard liquor', 'Small plates', 'Vegetarian options', 'Wine'], [], ['Alcohol', 'Beer', 'Cocktails', 'Food', 'Happy hour drinks', 'Happy hour food', 'Hard liquor', 'Wine'], ['Alcohol', 'Cocktails', 'Food', 'Food at bar', 'Hard liquor', 'Wine'], ['Coffee'], ['Coffee'], ['Alcohol', 'Beer', 'Cocktails', 'Coffee', 'Hard liq

In [14]:
file_path = os.path.join(os.getcwd(), "data/data_clean_normalize.csv")
df = pd.read_csv(file_path, encoding="utf-8")
df.head()

,id,name,address,lat,lon,poi_type,avg_stars,total_reviews,stay_time,normalize_stars_reviews,crowd,offerings,atmosphere,highlights,dining_options
0,0f9d2009-9436-46a4-b354-b0261898a39e,The Pub Coffee - Beer & Cocktail,"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thà...",10.829481,106.773785,"Cafe,Bar",4.9,181,30,0.755,Groups,"Alcohol, Beer, Cocktails, Coffee, Hard liquor","Casual, Cozy","Great beer selection, Great coffee, Live music...",Table service
1,02887955-963a-43ac-b0f7-355d7d7cfacf,Julieta,"C. Sta. Lucía, 9, Distrito Centro, 29008 Málag...",36.722011,-4.421780,Cafe,4.3,2053,30,0.661,"College students, Groups, Tourists","Alcohol, Beer, Coffee, Healthy options, Organi...","Casual, Cozy, Trendy","Great coffee, Great dessert, Great tea selection","Breakfast, Brunch, Lunch, Dessert, Seating, Ta..."
2,622c7643-30e8-4402-9b6c-b8407ff063e2,BAIETA Restaurant Saigon Thao Dien,"Nguyễn Văn Hưởng, Thảo Điền, 16/8, Thành phố H...",10.802978,106.727268,"Restaurant,Bar,Cocktail bar,Coffee shop,Family...",4.5,285,30,0.651,"Family-friendly, Groups, LGBTQ+ friendly, Tour...","Alcohol, Beer, Cocktails, Coffee, Happy hour d...","Casual, Cozy, Romantic","Great beer selection, Great cocktails, Great c...","Brunch, Lunch, Dinner, Catering, Counter servi..."
3,4f06908d-e9fa-4f6a-b1ae-c7d8882e2edf,Parroquia San Felipe Neri,"C. Guerrero, 6, Distrito Centro, 29012 Málaga,...",36.725554,-4.421321,"Catholic church,Tourist attraction",4.7,329,30,0.716,NaN,NaN,NaN,NaN,NaN
4,279dfce3-c227-4b58-b4ed-09197327a32a,BÀ BAR,"15 Đ. Nguyễn Cừ, Thảo Điền, Quận 2, Thành phố ...",10.803190,106.728381,"Cocktail bar,Cafe",4.8,182,30,0.725,"Groups, LGBTQ+ friendly, Transgender safespace","Alcohol, Beer, Cocktails, Food, Happy hour dri...",Casual,"Live music, Live performances","Seating, Table service"


In [110]:
df["crowd"] = [", ".join(c) for c in crowd]
df["offerings"] = [", ".join(c) for c in offerings]
df["atmosphere"] = [", ".join(c) for c in atmosphere]
df["highlights"] = [", ".join(c) for c in highlights]
df["dining_options"] = [", ".join(c) for c in dining_options]
df["children"] = [", ".join(c) for c in children]
df["accessibility"] = [", ".join(c) for c in accessibility]
df["popular_for"] = [", ".join(c) for c in popular_for]

In [16]:
df.head()

,id,name,address,lat,lon,poi_type,avg_stars,total_reviews,stay_time,normalize_stars_reviews,crowd,offerings,atmosphere,highlights,dining_options
0,0f9d2009-9436-46a4-b354-b0261898a39e,The Pub Coffee - Beer & Cocktail,"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thà...",10.829481,106.773785,"Cafe,Bar",4.9,181,30,0.755,[Groups],"Alcohol, Beer, Cocktails, Coffee, Hard liquor","Casual, Cozy","Great beer selection, Great coffee, Live music...",Table service
1,02887955-963a-43ac-b0f7-355d7d7cfacf,Julieta,"C. Sta. Lucía, 9, Distrito Centro, 29008 Málag...",36.722011,-4.421780,Cafe,4.3,2053,30,0.661,"[College students, Groups, Tourists]","Alcohol, Beer, Coffee, Healthy options, Organi...","Casual, Cozy, Trendy","Great coffee, Great dessert, Great tea selection","Breakfast, Brunch, Lunch, Dessert, Seating, Ta..."
2,622c7643-30e8-4402-9b6c-b8407ff063e2,BAIETA Restaurant Saigon Thao Dien,"Nguyễn Văn Hưởng, Thảo Điền, 16/8, Thành phố H...",10.802978,106.727268,"Restaurant,Bar,Cocktail bar,Coffee shop,Family...",4.5,285,30,0.651,"[Family-friendly, Groups, LGBTQ+ friendly, Tou...","Alcohol, Beer, Cocktails, Coffee, Happy hour d...","Casual, Cozy, Romantic","Great beer selection, Great cocktails, Great c...","Brunch, Lunch, Dinner, Catering, Counter servi..."
3,4f06908d-e9fa-4f6a-b1ae-c7d8882e2edf,Parroquia San Felipe Neri,"C. Guerrero, 6, Distrito Centro, 29012 Málaga,...",36.725554,-4.421321,"Catholic church,Tourist attraction",4.7,329,30,0.716,[],NaN,NaN,NaN,NaN
4,279dfce3-c227-4b58-b4ed-09197327a32a,BÀ BAR,"15 Đ. Nguyễn Cừ, Thảo Điền, Quận 2, Thành phố ...",10.803190,106.728381,"Cocktail bar,Cafe",4.8,182,30,0.725,"[Groups, LGBTQ+ friendly, Transgender safespace]","Alcohol, Beer, Cocktails, Food, Happy hour dri...",Casual,"Live music, Live performances","Seating, Table service"


In [ ]:
df.to_csv("data/data_final.csv", index=False)

# Thêm openingHours

In [ ]:
# openingHours từ raw_data

In [11]:
cur.execute('SELECT raw_data FROM "Poi"')
rows = cur.fetchall()
for poi in rows[:10]:
    print(poi)

RealDictRow([('raw_data', {'google': {'cid': '9137476420856649765', 'fid': '0x317527b418461fed:0x7eced66470ab1825', 'url': 'https://www.google.com/maps/search/?api=1&query=The%20Pub%20Coffee%20-%20Beer%20%26%20Cocktail&query_place_id=ChIJ7R9GGLQndTERJRircGTWzn4', 'city': 'Ho Chi Minh City', 'menu': None, 'rank': 1, 'kgmid': '/g/11fpkvlhz2', 'phone': '+84 974 188 353', 'price': None, 'state': 'Quận 9, Ho Chi Minh City', 'title': 'The Pub Coffee - Beer & Cocktail', 'emails': [], 'images': None, 'phones': [], 'street': '18A17 Tăng Nhơn Phú', 'address': '18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thành phố Hồ Chí Minh, Vietnam', 'orderBy': [], 'placeId': 'ChIJ7R9GGLQndTERJRircGTWzn4', 'reviews': [{'name': 'Thảo Thanh', 'text': 'Quán rộng rãi thoáng mát, giá cả sinh viên', 'stars': 5, 'rating': None, 'reviewId': 'Ci9DQUlRQUNvZENodHljRjlvT21WaFR6QTVTblptZDBWQmRISTVTSFptYm1STmFsRRAB', 'publishAt': 'a week ago', 'reviewUrl': 'https://www.google.com/maps/reviews/data=!4m8!14m7!1m6!2m5!1sCi9DQUl

## Xử lí xóa \u202f

In [44]:
def clean_opening_hours(data):
    cleaned = []
    for items in data:
        cleaned_items = []
        for i in items:
            cleaned_items.append({
                "day": i["day"],
                "hours": i["hours"].replace("\u202f", " ")
            })
        cleaned.append(cleaned_items)
    return cleaned


In [45]:
opening_hours = []

for poi in rows:
    raw_data = poi.get("raw_data") 
    google = raw_data.get("google")
    openingHours = google.get("openingHours")
    opening_hours.append(openingHours)

In [67]:
result =  clean_opening_hours(opening_hours)
print(result[0])
print(result[21])
print(result[22])

[{'day': 'Monday', 'hours': 'Open 24 hours'}, {'day': 'Tuesday', 'hours': 'Open 24 hours'}, {'day': 'Wednesday', 'hours': 'Open 24 hours'}, {'day': 'Thursday', 'hours': 'Open 24 hours'}, {'day': 'Friday', 'hours': 'Open 24 hours'}, {'day': 'Saturday', 'hours': 'Open 24 hours'}, {'day': 'Sunday', 'hours': 'Open 24 hours'}]
[{'day': 'Monday', 'hours': '8 to 11 PM'}, {'day': 'Tuesday', 'hours': '1:30 to 4 PM, 8 to 11 PM'}, {'day': 'Wednesday', 'hours': 'Closed'}, {'day': 'Thursday', 'hours': '1:30 to 4 PM, 8 to 11 PM'}, {'day': 'Friday', 'hours': '1:30 to 4 PM, 8 to 11 PM'}, {'day': 'Saturday', 'hours': '1:30 to 4 PM, 8 to 11 PM'}, {'day': 'Sunday', 'hours': 'Closed'}]
[{'day': 'Monday', 'hours': 'Closed'}, {'day': 'Tuesday', 'hours': '7:30 AM to 8 PM'}, {'day': 'Wednesday', 'hours': '7:30 AM to 8 PM'}, {'day': 'Thursday', 'hours': '7:30 AM to 8 PM'}, {'day': 'Friday', 'hours': '7:30 AM to 8 PM'}, {'day': 'Saturday', 'hours': '5:30 AM to 8:30 PM'}, {'day': 'Sunday', 'hours': '5:30 AM to 8

In [46]:
print(len(opening_hours))

1454


## Xử lí các trường hợp đặc biệt về thời gian cho openingHours

In [77]:
import re
from datetime import datetime

# ---- Helper: convert time string to 24h format ----
def to_24h(time_str, fallback_am_pm=None):
    time_str = time_str.strip()

    # If contains AM/PM → direct parse
    if "AM" in time_str.upper() or "PM" in time_str.upper():
        return datetime.strptime(time_str.upper(), "%I:%M %p").strftime("%H:%M") \
            if ":" in time_str else datetime.strptime(time_str.upper(), "%I %p").strftime("%H:%M")

    # If missing AM/PM, but fallback exists
    if fallback_am_pm:
        combined = f"{time_str} {fallback_am_pm}"
        try:
            return datetime.strptime(combined, "%I:%M %p").strftime("%H:%M") \
                if ":" in time_str else datetime.strptime(combined, "%I %p").strftime("%H:%M")
        except:
            pass

    # If totally unknown → assume 24h? But better skip
    return None


# ---- Main parser for hours ----
def parse_hours(hours_str):
    hours_str = hours_str.replace("\u202f", " ").strip()

    # Closed → skip
    if hours_str.lower() == "closed":
        return []

    # Open 24 hours
    if "open 24 hours" in hours_str.lower():
        return [{"start": "00:00", "end": "23:59"}]

    segments = [seg.strip() for seg in hours_str.split(",")]
    results = []

    for segment in segments:
        # Example segment: "1:30 to 4 PM"
        match = re.match(r"(.*) to (.*)", segment)
        if not match:
            continue

        start_raw, end_raw = match.groups()
        start_raw = start_raw.strip()
        end_raw = end_raw.strip()

        # Determine fallback AM/PM from end if start lacks AM/PM
        fallback = None
        if "AM" in end_raw.upper():
            fallback = "AM"
        if "PM" in end_raw.upper():
            fallback = "PM"

        start_24 = to_24h(start_raw, fallback)
        end_24 = to_24h(end_raw)

        if start_24 and end_24:
            results.append({
                "start": start_24,
                "end": end_24
            })

    return results


# ---- Main function to process full opening_hours array ----
def normalize_opening_hours(opening_hours):
    normalized = []

    for item in opening_hours:
        day = item["day"]
        hours = parse_hours(item["hours"])

        if hours:  # Skip closed days
            normalized.append({"day": day, "hours": hours})

    return normalized

def clean_opening_hours_special(data):
    cleaned_special = []
    for d in data:
        if len(d) == 0:
            cleaned_special.append([])
        else:
            item = normalize_opening_hours(d)
            cleaned_special.append(item)
    return cleaned_special


In [75]:
print(normalize_opening_hours(result[0]))
print(normalize_opening_hours(result[21]))
print(normalize_opening_hours(result[22]))


[{'day': 'Monday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Tuesday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Wednesday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Thursday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Friday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Saturday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Sunday', 'hours': [{'start': '00:00', 'end': '23:59'}]}]
[{'day': 'Monday', 'hours': [{'start': '20:00', 'end': '23:00'}]}, {'day': 'Tuesday', 'hours': [{'start': '13:30', 'end': '16:00'}, {'start': '20:00', 'end': '23:00'}]}, {'day': 'Thursday', 'hours': [{'start': '13:30', 'end': '16:00'}, {'start': '20:00', 'end': '23:00'}]}, {'day': 'Friday', 'hours': [{'start': '13:30', 'end': '16:00'}, {'start': '20:00', 'end': '23:00'}]}, {'day': 'Saturday', 'hours': [{'start': '13:30', 'end': '16:00'}, {'start': '20:00', 'end': '23:00'}]}]
[{'day': 'Tuesday', 'hours': [{'start': '07:30', 

In [78]:
clean_opening_hours_special(result)

[[{'day': 'Monday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
  {'day': 'Tuesday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
  {'day': 'Wednesday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
  {'day': 'Thursday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
  {'day': 'Friday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
  {'day': 'Saturday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
  {'day': 'Sunday', 'hours': [{'start': '00:00', 'end': '23:59'}]}],
 [{'day': 'Monday', 'hours': [{'start': '08:00', 'end': '20:30'}]},
  {'day': 'Tuesday', 'hours': [{'start': '08:00', 'end': '20:30'}]},
  {'day': 'Wednesday', 'hours': [{'start': '08:00', 'end': '20:30'}]},
  {'day': 'Thursday', 'hours': [{'start': '08:00', 'end': '20:30'}]},
  {'day': 'Friday', 'hours': [{'start': '08:00', 'end': '20:30'}]},
  {'day': 'Saturday', 'hours': [{'start': '09:00', 'end': '20:30'}]},
  {'day': 'Sunday', 'hours': [{'start': '09:00', 'end': '20:30'}]}],
 [{'day': 'Tuesday', 'hours': 

In [79]:
len(clean_opening_hours_special(result))

1454

In [80]:
file_path = os.path.join(os.getcwd(), "data/data_final.csv")
df = pd.read_csv(file_path, encoding="utf-8")
df.head(1)

,id,name,address,lat,lon,poi_type,avg_stars,total_reviews,stay_time,normalize_stars_reviews,crowd,offerings,atmosphere,highlights,dining_options,children,accessibility,popular_for
0,0f9d2009-9436-46a4-b354-b0261898a39e,The Pub Coffee - Beer & Cocktail,"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thà...",10.829481,106.773785,"Cafe,Bar",4.9,181,30,0.755,Groups,"Alcohol, Beer, Cocktails, Coffee, Hard liquor","Casual, Cozy","Great beer selection, Great coffee, Live music...",Table service,NaN,NaN,NaN


In [82]:
df['opening_hours'] = clean_opening_hours_special(result)

In [83]:
df.head(1)

,id,name,address,lat,lon,poi_type,avg_stars,total_reviews,stay_time,normalize_stars_reviews,crowd,offerings,atmosphere,highlights,dining_options,children,accessibility,popular_for,opening_hours
0,0f9d2009-9436-46a4-b354-b0261898a39e,The Pub Coffee - Beer & Cocktail,"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thà...",10.829481,106.773785,"Cafe,Bar",4.9,181,30,0.755,Groups,"Alcohol, Beer, Cocktails, Coffee, Hard liquor","Casual, Cozy","Great beer selection, Great coffee, Live music...",Table service,NaN,NaN,NaN,"[{'day': 'Monday', 'hours': [{'start': '00:00'..."


In [ ]:
df.to_csv("data/new_data_final.csv", index=False)

# Test time bằng python

In [52]:
from datetime import datetime

now = datetime.now()

# Lấy giờ và phút
gio_phut = now.strftime("%H:%M")

# Lấy thứ (tiếng Anh)
thu = now.strftime("%A")

print("Thứ:", thu)
print("Giờ và phút:", gio_phut)

Thứ: Friday
Giờ và phút: 15:16


# Clean opening_hours

In [1]:
opening_hours = [{'day': 'Monday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Tuesday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Wednesday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Thursday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Friday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Saturday', 'hours': [{'start': '00:00', 'end': '23:59'}]}, {'day': 'Sunday', 'hours': [{'start': '00:00', 'end': '23:59'}]}]